In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_125_Punjabi_Bagh_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,218.04,312.56,0.86,66.53,36.08,27.39,18.71,1.31,8.09,...,NaN,13.81,74.04,0.65,163.88,NaN,0.0,32.93,NaN,0.02
1,2024-01-02,199.46,281.16,0.82,66.92,36.25,25.09,11.93,1.43,9.42,...,NaN,13.66,71.19,0.62,160.47,NaN,0.0,45.97,NaN,0.01
2,2024-01-03,236.59,326.28,13.72,66.65,46.47,30.11,4.92,1.61,10.51,...,NaN,13.53,82.88,0.50,162.42,NaN,0.0,25.52,NaN,0.01
3,2024-01-04,250.55,342.07,32.24,61.89,59.13,29.32,5.96,1.04,9.21,...,NaN,13.68,84.51,0.65,192.27,NaN,0.0,14.42,NaN,0.00
4,2024-01-05,196.11,297.15,33.75,61.89,60.36,29.08,2.32,1.43,9.07,...,NaN,14.24,87.58,0.42,193.43,NaN,0.0,11.96,NaN,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,160.70,209.78,21.02,60.88,49.48,60.54,28.72,1.49,9.70,...,NaN,16.36,87.49,0.44,187.24,NaN,0.0,9.93,NaN,-0.00
362,2024-12-28,106.45,137.68,26.50,51.80,49.11,54.85,39.39,1.57,5.82,...,NaN,16.43,90.93,0.34,193.34,NaN,0.0,14.14,NaN,-0.01
363,2024-12-29,97.33,136.54,6.72,41.20,27.38,46.15,32.18,0.76,18.10,...,NaN,15.99,88.18,0.64,202.57,NaN,0.0,46.01,NaN,-0.00
364,2024-12-30,99.17,139.88,6.21,41.99,27.38,39.72,32.53,0.65,23.03,...,NaN,14.85,83.52,0.63,196.77,NaN,0.0,40.17,NaN,-0.01


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 22)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Benzene (µg/m³)', 'Toluene (µg/m³)', 'Xylene (µg/m³)', 'MP-Xylene (µg/m³)', 'BP (mmHg)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp        0
PM2.5 (µg/m³)    0
PM10 (µg/m³)     0
NO (µg/m³)       0
NO2 (µg/m³)      0
NOx (ppb)        0
NH3 (µg/m³)      0
SO2 (µg/m³)      0
CO (mg/m³)       0
Ozone (µg/m³)    0
AT (°C)          0
RH (%)           0
WS (m/s)         0
WD (deg)         0
TOT-RF (mm)      0
SR (W/mt2)       0
VWS (m/s)        0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 17)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         218.04        312.56        0.86        66.53   
1  2024-01-02         199.46        281.16        0.82        66.92   
2  2024-01-03         236.59        326.28       13.72        66.65   
3  2024-01-04         250.55        342.07       32.24        61.89   
4  2024-01-05         196.11        297.15       33.75        61.89   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  AT (°C)  \
0      36.08        27.39        18.71        1.31           8.09    13.81   
1      36.25        25.09        11.93        1.43           9.42    13.66   
2      46.47        30.11         4.92        1.61          10.51    13.53   
3      59.13        29.32         5.96        1.04           9.21    13.68   
4      60.36        29.08         2.32        1.43           9.07    14.24   

   RH (%)  WS (m/s)  WD (deg)  TOT-RF (mm)  SR (W/mt2)  VWS (m/s)  
0   74.04    

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),TOT-RF (mm),SR (W/mt2),VWS (m/s)
0,2024-01-01,1.606741,0.960642,-1.082375,1.363874,0.219422,1.226834,0.064296,0.863329,-1.264118,-1.740715,0.837531,0.429961,-1.224919,0.0,-1.559535,1.496596
1,2024-01-02,1.341719,0.657302,-1.086223,1.383721,0.228324,0.869430,-0.888064,1.204589,-1.173311,-1.759619,0.655534,0.272333,-1.524856,0.0,-0.930440,0.712971
2,2024-01-03,1.871336,1.093184,0.154957,1.369981,0.763489,1.649504,-1.872731,1.716479,-1.098891,-1.776001,1.402040,-0.358181,-1.353338,0.0,-1.917019,0.712971
3,2024-01-04,2.070459,1.245723,1.936869,1.127753,1.426423,1.526743,-1.726646,0.095494,-1.187649,-1.757098,1.506130,0.429961,1.272214,0.0,-2.452521,-0.070655
4,2024-01-05,1.293935,0.811773,2.082154,1.127753,1.490832,1.489449,-2.237942,1.204589,-1.197208,-1.686526,1.702175,-0.778524,1.374246,0.0,-2.571200,0.712971
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,0.788851,-0.032265,0.857330,1.076356,0.921106,-0.031074,1.470361,1.375219,-1.154194,-1.419359,1.696428,-0.673438,0.829785,0.0,-2.669135,-0.070655
362,2024-12-28,0.015037,-0.728788,1.384592,0.614292,0.901731,-0.031074,0.128910,1.602726,-1.419103,-1.410538,1.916101,-1.198867,1.366329,0.0,-2.466030,-0.854280
363,2024-12-29,-0.115050,-0.739801,-0.518552,0.074879,-0.236150,-0.031074,1.956373,-0.700780,-0.580681,-1.465987,1.740490,0.377418,2.178183,0.0,-0.928510,-0.070655
364,2024-12-30,-0.088804,-0.707534,-0.567622,0.115080,-0.236150,-0.031074,2.005536,-1.013602,-0.244083,-1.609652,1.442910,0.324875,1.668026,0.0,-1.210252,-0.854280


In [10]:
df.to_excel('punjabibagh2024.xlsx', index=False)